# 第 03 天：IC 基础

> 来自《30 天因子研究计划》第 3 天  
> 主题：IC 基础  
> 必做：Pearson IC  
> 选做：Spearman Rank IC  
> 目标产出：计算样例 IC

---

## 0. 今天你要真正学会什么？

第 1 天我们知道了因子投资是在拆解收益来源。  
第 2 天我们把价格变成了未来收益标签。  
第 3 天终于要问一个因子研究里最经典的问题：

> 在同一个交易日里，因子值更高的股票，未来收益是不是也更高？

这个问题的标准量化工具就是 IC。

IC 的英文是 `Information Coefficient`，常译为信息系数。它本质上就是：


因子值 与 未来收益 的截面相关系数


学完以后，你应该能回答：

1. IC 到底在衡量什么？
2. Pearson IC 和 Spearman Rank IC 有什么区别？
3. 为什么 IC 要按日期在截面上计算，而不是把所有样本混在一起算？
4. IC 为正、为负、接近 0 分别意味着什么？
5. 如何用 Python 计算单日 IC 和每日 IC 序列？

一句话版：

> IC 是因子研究的第一把尺子，用来衡量“今天的因子排序”和“未来的收益排序”之间有没有关系。

---

## 1. 先建立直觉：IC 像一次班级排名预测

想象一个班里有 100 个学生。

今天你根据“平时作业分”给学生排了一个名次。  
20 天后，学生参加考试，你拿到真实考试分数。

你想知道：

> 平时作业分高的人，考试分数是不是也更高？

这就是相关性问题。

在因子研究里：

| 学校类比 | 因子研究 |
| --- | --- |
| 学生 | 股票 |
| 平时作业分 | 因子值 |
| 20 天后考试分数 | 未来 20 日收益 |
| 排名预测能力 | IC |

如果因子值越高，未来收益越高，IC 通常为正。  
如果因子值越高，未来收益越低，IC 通常为负。  
如果二者没什么关系，IC 接近 0。

---

## 2. IC 的正式定义

在某个日期 `t`，有很多股票 `i = 1, 2, ..., N`。

每只股票有：

- 因子值：`factor_{t,i}`
- 未来收益：`return_{t,i}`

那么这一天的 IC 是：


IC_t = corr(factor_{t,*}, future_return_{t,*})


其中 `*` 表示这一天所有股票的截面。

关键点：


IC 是按日期计算的截面相关性。


不是这样：


把所有日期、所有股票混在一起算一个相关性


而是这样：


2024-01-02 算一个 IC
2024-01-03 算一个 IC
2024-01-04 算一个 IC
...
最后得到一条 IC 时间序列


后面第 4 天的 ICIR，就是在这条 IC 时间序列上分析稳定性。

---

## 3. Pearson IC 与 Spearman Rank IC

### 3.1 Pearson IC：看线性相关

Pearson 相关系数衡量的是两个变量之间的线性关系。


Pearson IC = corr(factor_value, future_return)


它关心数值大小。

如果因子值从 1 到 2，未来收益也大致线性变高，Pearson IC 会比较高。

### 3.2 Spearman Rank IC：看排序相关

Spearman 相关系数先把数值变成排名，再计算 Pearson 相关。


Spearman Rank IC = corr(rank(factor_value), rank(future_return))


它更关心排序，不太在意具体数值间隔。

举个例子：


因子值：1, 2, 100
排名：  1, 2, 3


100 这个极端值会强烈影响 Pearson，但对 Spearman 来说，它只是排名第 3。

### 3.3 什么时候看哪个？

| 指标 | 更关心 | 优点 | 风险 |
| --- | --- | --- | --- |
| Pearson IC | 数值线性关系 | 对真实线性强度敏感 | 容易被极端值影响 |
| Spearman Rank IC | 排序关系 | 更稳健，更贴近选股排序 | 会丢掉数值间距信息 |

因子研究里经常更重视 Rank IC，因为很多时候我们不是精确预测收益率，而是想知道：

> 哪些股票更值得排在前面？

---

## 4. 准备 Python 环境


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["axes.grid"] = True

rng = np.random.default_rng(20260705)


如果缺包，可以先安装：


In [ ]:
pip install numpy pandas matplotlib


---

## 5. 构造一份样例因子数据

我们先用模拟数据，原因很简单：

> 模拟世界里我们知道因子有没有用，更容易理解 IC 的含义。

### 5.1 构造股票池和日期


In [ ]:
dates = pd.bdate_range("2024-01-02", periods=160)
tickers = [f"Stock_{i:03d}" for i in range(300)]

len(dates), len(tickers)


### 5.2 构造一个“有一点点有效”的因子

真实因子通常很弱。我们故意让因子和未来收益只有一点关系。


In [ ]:
records = []

for date in dates:
    # 原始因子值：大部分是随机的
    factor = rng.normal(0, 1, size=len(tickers))

    # 横截面噪声：真实世界里未来收益大部分是噪声
    noise = rng.normal(0, 0.06, size=len(tickers))

    # 未来20日收益：让它和因子值有一点点正相关
    future_20d_ret = 0.015 * factor + noise

    for ticker, f, r in zip(tickers, factor, future_20d_ret):
        records.append((date, ticker, f, r))

sample = pd.DataFrame(
    records,
    columns=["date", "ticker", "factor_value", "future_20d_ret"]
)

sample.head()


这张表就是第 2 天目标产出的下一步形态：


date | ticker | factor_value | future_20d_ret


---

## 6. 计算某一天的样例 IC

### 6.1 取出单日截面


In [ ]:
one_date = sample["date"].iloc[0]
one_day = sample.loc[sample["date"] == one_date].copy()

one_date, one_day.head()


这一天有 300 只股票。  
我们要在这 300 只股票之间计算因子值和未来收益的相关性。

### 6.2 Pearson IC


In [ ]:
pearson_ic = one_day["factor_value"].corr(one_day["future_20d_ret"], method="pearson")
print("单日 Pearson IC:", round(pearson_ic, 4))


### 6.3 Spearman Rank IC


In [ ]:
rank_ic = one_day["factor_value"].corr(one_day["future_20d_ret"], method="spearman")
print("单日 Spearman Rank IC:", round(rank_ic, 4))


### 6.4 手工理解 Rank IC

Spearman 相关可以理解为：


先排名，再算 Pearson


In [ ]:
one_day["factor_rank"] = one_day["factor_value"].rank()
one_day["return_rank"] = one_day["future_20d_ret"].rank()

manual_rank_ic = one_day["factor_rank"].corr(one_day["return_rank"], method="pearson")
print("手工排名后计算的 Rank IC:", round(manual_rank_ic, 4))


这个结果应该和 `method="spearman"` 一致。

---

## 7. 画图理解 IC

### 7.1 散点图


In [ ]:
plt.scatter(one_day["factor_value"], one_day["future_20d_ret"], alpha=0.55)
plt.axhline(0, color="black", linewidth=1)
plt.axvline(0, color="black", linewidth=1)
plt.title(f"One-day cross-section: Pearson IC = {pearson_ic:.3f}")
plt.xlabel("Factor value")
plt.ylabel("Future 20D return")
plt.show()


如果点云略微从左下往右上倾斜，说明 IC 为正。

注意：真实因子的散点图通常不会特别漂亮。  
因子研究靠的是大样本里微弱但稳定的统计关系，不是每一天都清清楚楚。

### 7.2 排名散点图


In [ ]:
plt.scatter(one_day["factor_rank"], one_day["return_rank"], alpha=0.55)
plt.title(f"One-day rank relation: Rank IC = {rank_ic:.3f}")
plt.xlabel("Factor rank")
plt.ylabel("Future return rank")
plt.show()


Rank IC 更接近“排序能力”。

如果你未来要做分组回测或多头组合，排序能力通常比精确收益预测更重要。

---

## 8. 计算每日 IC 序列

单日 IC 只是一个切片。

真正要看因子有没有用，需要对每个交易日都算 IC。

### 8.1 写一个 IC 函数


In [ ]:
def calc_daily_ic(
    data: pd.DataFrame,
    factor_col: str,
    return_col: str,
    method: str = "pearson",
    min_count: int = 30,
) -> pd.Series:
    """
    按日期计算截面 IC。

    data 必须至少包含 date、factor_col、return_col 三列。
    method 可选 pearson 或 spearman。
    min_count 用来避免股票数量太少时相关性不可靠。
    """
    ic_values = {}

    for date, group in data.groupby("date"):
        g = group[[factor_col, return_col]].dropna()

        if len(g) < min_count:
            ic_values[date] = np.nan
            continue

        ic_values[date] = g[factor_col].corr(g[return_col], method=method)

    return pd.Series(ic_values, name=f"{method}_ic").sort_index()


### 8.2 计算 Pearson IC 序列


In [ ]:
daily_pearson_ic = calc_daily_ic(
    sample,
    factor_col="factor_value",
    return_col="future_20d_ret",
    method="pearson"
)

daily_pearson_ic.head()


### 8.3 计算 Rank IC 序列


In [ ]:
daily_rank_ic = calc_daily_ic(
    sample,
    factor_col="factor_value",
    return_col="future_20d_ret",
    method="spearman"
)

daily_rank_ic.head()


### 8.4 汇总结果


In [ ]:
ic_summary = pd.DataFrame({
    "mean": [daily_pearson_ic.mean(), daily_rank_ic.mean()],
    "std": [daily_pearson_ic.std(), daily_rank_ic.std()],
    "positive_ratio": [(daily_pearson_ic > 0).mean(), (daily_rank_ic > 0).mean()],
}, index=["Pearson IC", "Rank IC"])

ic_summary


常见解释：

- `mean`：平均 IC，越大代表平均预测关系越强。
- `std`：IC 波动，越大代表稳定性越差。
- `positive_ratio`：IC 为正的日期占比。

---

## 9. IC 的数值该怎么解读？

IC 通常不会很大。

在真实股票横截面里，一个日频或月频因子如果长期 Rank IC 均值有 `0.02` 到 `0.05`，就已经值得认真研究。

粗略参考：

| IC 均值 | 直觉 |
| ---: | --- |
| 接近 0 | 几乎没有稳定预测关系 |
| 0.01 - 0.02 | 很弱，需要看稳定性和成本 |
| 0.02 - 0.05 | 有研究价值 |
| 0.05 以上 | 很强，要警惕未来函数或样本偏差 |
| 显著为负 | 可能反向有效 |

这里有个很重要的提醒：

> IC 小不代表没用。只要方向稳定，组合层面可能仍然能积累优势。

---

## 10. 为什么不能把所有日期混在一起算？

很多初学者会写：


In [ ]:
pooled_corr = sample["factor_value"].corr(sample["future_20d_ret"])
print("把所有日期混在一起算的相关性:", round(pooled_corr, 4))


这个数字不是完全没意义，但它不是标准 IC。

标准 IC 关心的是：


每一天，在当天股票池内部，因子能不能区分未来收益高低。


如果你把所有日期混在一起，可能会混入：

- 市场整体涨跌的时间效应
- 不同日期波动率不同
- 股票池结构变化
- 样本分布漂移

所以标准流程是：


先每天算截面 IC
再分析 IC 时间序列


---

## 11. 缺失值和极端值

真实数据里会有缺失值和极端值。

### 11.1 加一点脏数据


In [ ]:
dirty = sample.copy()

dirty.loc[dirty.sample(frac=0.01, random_state=1).index, "factor_value"] = np.nan
dirty.loc[dirty.sample(frac=0.01, random_state=2).index, "future_20d_ret"] = np.nan

outlier_idx = dirty.sample(frac=0.002, random_state=3).index
dirty.loc[outlier_idx, "factor_value"] = dirty.loc[outlier_idx, "factor_value"] * 20

dirty[["factor_value", "future_20d_ret"]].describe()


### 11.2 比较 Pearson 和 Rank IC


In [ ]:
dirty_pearson_ic = calc_daily_ic(dirty, "factor_value", "future_20d_ret", method="pearson")
dirty_rank_ic = calc_daily_ic(dirty, "factor_value", "future_20d_ret", method="spearman")

pd.DataFrame({
    "clean_pearson": daily_pearson_ic.describe(),
    "dirty_pearson": dirty_pearson_ic.describe(),
    "clean_rank": daily_rank_ic.describe(),
    "dirty_rank": dirty_rank_ic.describe(),
})


通常 Rank IC 对极端值更稳健。  
这也是因子研究中 Rank IC 很常见的原因。

---

## 12. IC 的方向：正向、反向和无效

### 12.1 正向因子

因子值越高，未来收益越高，IC 为正。

例子：


高质量公司未来更抗跌，质量因子可能为正。


### 12.2 反向因子

因子值越高，未来收益越低，IC 为负。

这不一定是坏事。

如果一个因子长期 IC 为负，你可以把因子方向反过来：


new_factor = -old_factor


### 12.3 无效因子

IC 均值接近 0，而且正负很随机，说明它可能没有稳定预测能力。

但不要只看均值，还要看：

- 不同市场阶段
- 不同股票池
- 不同持有期
- 是否行业或市值暴露造成
- 样本外是否仍然有效

---

## 13. 今日目标产出：计算样例 IC

下面把今天的核心流程整理成一个最小函数。


In [ ]:
def ic_report(
    data: pd.DataFrame,
    factor_col: str = "factor_value",
    return_col: str = "future_20d_ret",
) -> pd.DataFrame:
    pearson = calc_daily_ic(data, factor_col, return_col, method="pearson")
    rank = calc_daily_ic(data, factor_col, return_col, method="spearman")

    report = pd.DataFrame({
        "mean_ic": [pearson.mean(), rank.mean()],
        "std_ic": [pearson.std(), rank.std()],
        "positive_ratio": [(pearson > 0).mean(), (rank > 0).mean()],
        "count": [pearson.count(), rank.count()],
    }, index=["Pearson IC", "Rank IC"])

    return report


report = ic_report(sample)
report


这就是第 3 天的目标产出：  
你能对一份 `date, ticker, factor, future_return` 数据计算样例 IC。

---

## 14. 今天的知识图谱


In [ ]:
mindmap
  root((IC基础))
    输入数据
      date
      ticker
      factor_value
      future_return
    PearsonIC
      数值相关
      线性关系
      易受极端值影响
    RankIC
      排名相关
      Spearman
      更贴近选股排序
      更稳健
    计算方式
      按日期分组
      每日截面相关
      得到IC序列
    解释
      正IC
      负IC
      接近0
      正IC占比
    风险点
      全样本混算
      缺失值
      极端值
      未来函数
      样本太少


文本版：


IC 基础
├── 输入
│   ├── 因子值
│   └── 未来收益标签
├── Pearson IC
│   ├── 线性相关
│   └── 对极端值敏感
├── Rank IC
│   ├── 排名相关
│   └── 更适合选股排序
├── 计算流程
│   ├── 按日期分组
│   ├── 单日截面相关
│   └── IC 时间序列
└── 解读
    ├── 均值
    ├── 标准差
    └── 正 IC 占比


---

## 15. 初学者最容易踩的 7 个坑

### 坑 1：把所有日期混在一起算相关性

这不是标准 IC。标准 IC 是每天算一个截面相关。

### 坑 2：忘记因子方向

有些指标越小越好，比如 PB、PE、波动率。  
方向没处理好，IC 可能会反着来。

### 坑 3：只看 Pearson IC

Pearson 会被极端值影响。建议同时看 Rank IC。

### 坑 4：样本太少也算 IC

如果某天只有几只股票，相关性很不稳定。可以设置 `min_count`。

### 坑 5：把 IC 当作收益率

IC 是相关性，不是策略收益。IC 高不代表直接赚钱，还要看分组回测、成本和组合构建。

### 坑 6：看到单日 IC 负就否定因子

单日 IC 噪声很大，要看长期均值和稳定性。

### 坑 7：未来收益标签错位

如果第 2 天的标签错了，第 3 天的 IC 会认真回答一个错误问题。

---

## 16. 今天的动手作业

### 作业 A：解释 IC

用自己的话回答：

1. IC 为什么是“截面相关”？
2. Pearson IC 和 Rank IC 分别看什么？
3. 为什么 Rank IC 更适合很多选股场景？

### 作业 B：运行样例代码

运行本文所有 Python 代码，记录：

- 单日 Pearson IC
- 单日 Rank IC
- IC 均值
- IC 标准差
- 正 IC 占比

### 作业 C：改造因子强度

把这行：


In [ ]:
future_20d_ret = 0.015 * factor + noise


改成：


In [ ]:
future_20d_ret = 0.005 * factor + noise
future_20d_ret = 0.030 * factor + noise
future_20d_ret = -0.015 * factor + noise


观察 IC 如何变化。

### 作业 D：故意加入极端值

把少数因子值乘以 50，再比较 Pearson IC 和 Rank IC。  
看看哪个指标更稳定。

### 作业 E：写自己的 IC 函数

要求：


In [ ]:
def calc_ic(data, factor_col, return_col, method):
    ...


输出每日 IC 序列。

---

## 17. 自测题

### 题 1

IC 的核心输入是哪两列？

答案：因子值和未来收益标签。

### 题 2

IC 应该按什么维度分组计算？

答案：按日期分组，在每个日期的股票截面上计算。

### 题 3

Pearson IC 和 Rank IC 的最大区别是什么？

答案：Pearson 看原始数值的线性相关，Rank IC 看排名相关。

### 题 4

一个因子长期 Rank IC 为 -0.03，一定没用吗？

答案：不一定。它可能反向有效，可以尝试把因子方向取负后再检验。

### 题 5

IC 均值很高是否一定可信？

答案：不一定。异常高的 IC 要警惕未来函数、样本污染、标签错位或过拟合。

---

## 18. 今日复盘模板


第 03 天复盘：IC 基础

1. 我今天理解的 IC：

2. Pearson IC 的特点：

3. Rank IC 的特点：

4. 我计算出的单日 IC：

5. 我计算出的 IC 均值：

6. 我觉得最容易出错的地方：

7. 明天学习 ICIR 前，我需要回顾：


---

## 19. 明天预告：ICIR

今天我们得到了一条 IC 时间序列。  
明天会问：

> 这条 IC 序列是否稳定？

平均 IC 高但波动巨大，可能不如平均 IC 稍低但更稳定的因子。  
这就是 ICIR 要解决的问题。

---

## 20. 一句话收尾

IC 不是在问“这个因子今天能不能精准预测收益率”，而是在问：

> 在一批股票里，因子排序和未来收益排序有没有稳定关系？

只要你把这个问题想清楚，因子研究的大门就又打开了一层。

---

## 21. 仅供学习的提醒

本文所有示例使用模拟数据，仅用于解释 IC 的计算方法和直觉，不构成任何投资建议。真实研究还需要处理复权、停牌、退市、行业市值中性化、交易成本和样本外验证。

---

# 统一高质量增强模块

> 本增强模块用于把第 03 天课程统一提升到第 1-2 天那种“能直接学习、能直接运行、能直接复盘”的密度。前面的正文保留；下面是更完整的学习版。

## A. 今日任务重新聚焦

- 主题：IC基础
- 必做：Pearson IC
- 选做：Spearman Rank IC
- 目标产出：计算样例IC

今天真正要练成的不是“知道一个名词”，而是能把这个主题放进完整因子研究流水线：


原始数据
  ↓
因子构造
  ↓
预处理和对齐
  ↓
IC / ICIR / 分层回测
  ↓
形成可复用模块


你学习时可以一直问自己三句话：

1. 这个因子在经济含义上解释什么？
2. 这个因子在代码里如何被严格计算？
3. 这个因子是否真的经得起检验，而不是只在故事里成立？

## B. 一个更生动的直觉案例

你每天给股票排一个名次，20 天后看未来收益名次。IC 就是在问：你今天的排名和未来收益排名到底有多像。

这个例子背后的关键直觉是：

> IC 衡量的是截面排序能力，不是策略收益。

因子研究不是把金融名词翻译成代码，而是把一个投资假设拆成可以被验证、被复现、被质疑的实验。

## C. 今日知识骨架


IC基础
├── 输入数据
│   ├── 行情 / 财务 / 行业 / 市值等基础字段
│   └── 明确每个字段在当时是否可得
├── 因子定义
│   ├── 写清楚公式
│   ├── 写清楚方向
│   └── 写清楚缺失和异常值处理
├── 因子检验
│   ├── Rank IC
│   ├── ICIR
│   └── 分层回测
└── 目标产出
    └── 计算样例IC


## D. 完整 Python 实验

下面这段代码是一个自包含实验。你可以单独复制到 Notebook 里运行。它的目的不是模拟真实市场，而是把今天主题的计算口径、方向、检查方法串起来。


In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(103)
dates = pd.bdate_range("2024-01-02", periods=80)
tickers = [f"S{i:03d}" for i in range(200)]
rows = []
for d in dates:
    factor = rng.normal(0, 1, len(tickers))
    ret = 0.02 * factor + rng.normal(0, 0.06, len(tickers))
    rows.extend(zip([d] * len(tickers), tickers, factor, ret))
df = pd.DataFrame(rows, columns=["date", "ticker", "factor", "future_ret"])

daily_ic = df.groupby("date").apply(
    lambda g: g["factor"].corr(g["future_ret"], method="spearman"),
    include_groups=False
)
print(pd.Series({
    "mean_rank_ic": daily_ic.mean(),
    "std_rank_ic": daily_ic.std(),
    "positive_ratio": (daily_ic > 0).mean(),
}).round(4))


## E. 产出验收标准

完成今天课程后，你的 `计算样例IC` 至少应该满足：

1. 字段命名清晰，能看出日期、股票、因子值和标签含义。
2. 因子方向明确：值越大到底代表越好、越便宜、越强，还是越低风险。
3. 缺失值和异常值有处理口径，不把未知伪装成 0。
4. 至少有一段可重复运行的 Python 实验验证核心逻辑。
5. 能用 IC、ICIR 或分层回测中的至少一种方法做初步检查。
6. 能解释这个因子在真实研究里可能失效的原因。

如果这些检查没有过，不要急着进入下一天。因子研究里很多错误不是模型问题，而是最开始的口径、方向、对齐、缺失值处理出了问题。

## F. 常见坑深挖

### 坑 1：只记公式，不检查数据可得时点

这不是小问题。它会让你的因子看起来更有效，或者让真正有效的因子被误杀。处理方式是：先写清楚口径，再用代码抽样检查。
### 坑 2：因子方向写反，却直接进入 IC 和回测

这不是小问题。它会让你的因子看起来更有效，或者让真正有效的因子被误杀。处理方式是：先写清楚口径，再用代码抽样检查。
### 坑 3：把模拟数据里的漂亮结果当成真实市场规律

这不是小问题。它会让你的因子看起来更有效，或者让真正有效的因子被误杀。处理方式是：先写清楚口径，再用代码抽样检查。
### 坑 4：忽略缺失值、极端值和样本边界

这不是小问题。它会让你的因子看起来更有效，或者让真正有效的因子被误杀。处理方式是：先写清楚口径，再用代码抽样检查。
### 坑 5：只看单一指标，不做交叉验证

这不是小问题。它会让你的因子看起来更有效，或者让真正有效的因子被误杀。处理方式是：先写清楚口径，再用代码抽样检查。
### 坑 6：没有把目标产出封装成可复用函数

这不是小问题。它会让你的因子看起来更有效，或者让真正有效的因子被误杀。处理方式是：先写清楚口径，再用代码抽样检查。

## G. 强化练习

### 作业 A

用自己的话写出 `IC基础` 的一句话定义，并标明它属于收益、风险、估值、质量、技术、流动性还是预处理模块。
### 作业 B

运行完整实验代码，记录输出结果，并解释每一列结果的金融含义。
### 作业 C

故意把因子方向取反，再重新计算结果，观察 IC 或分组表现如何变化。
### 作业 D

加入 5% 缺失值或 1% 极端值，测试你的处理逻辑是否仍然稳健。
### 作业 E

把今天的 `计算样例IC` 保存成一个可以被后续课程调用的函数或表格。

## H. 面试式自测

### 问：这个主题在因子研究流水线里处于哪一步？

答：它对应 `计算样例IC`，用于把原始数据转成后续 IC、ICIR、分层回测或多因子合成可以使用的中间产物。
### 问：最容易出现未来函数的地方在哪里？

答：通常出现在使用未来才披露的数据、未来价格、未来收益标签错位，或把全样本统计量用于历史截面。
### 问：为什么不能只看一个漂亮结果？

答：因为单次结果可能来自样本偶然、极端值、行业暴露、市值暴露或参数过拟合，需要多角度验证。
### 问：如何判断今天产出的模块可以进入下一步？

答：至少通过字段检查、方向检查、缺失异常检查、抽样手工验证和一个简单统计检验。

## I. 今日复盘模板


第 03 天复盘：IC基础

1. 今天我能用一句话解释的核心概念：

2. 今天最重要的公式：

3. 代码里最容易写错的地方：

4. 我检查因子方向的方法：

5. 我检查缺失值和异常值的方法：

6. 如果把这个模块放进真实研究，我还缺什么数据：

7. 今天留下的一个问题：


## J. 和下一课的连接

下一课会继续沿着这条链路推进：前一天产出的字段或模块，会成为后一天检验、扩展或组合的输入。学习时不要把每天割裂开；真正的因子研究是一条流水线。

---

## K. 学习提醒

这一份课程仍然是教学材料，示例数据是模拟数据。真实研究需要处理真实数据源、可得时点、复权、停牌、交易成本、行业和市值暴露、样本外验证。
